# Day 63: Training Neural Networks — The Complete Guide
## Loss Functions, Backpropagation, Optimizers & Overfitting Prevention

---

# PART 1: THEORY

## 1. The Training Loop — How Every Neural Network Learns

Every neural network, from a 3-layer MNIST classifier to GPT-4, trains using the same 3-step loop repeated for every epoch:

```
FOR EACH EPOCH:
    1. FORWARD PASS:    Input flows through layers -> Prediction
    2. CALCULATE LOSS:  Compare prediction to ground truth -> Error number
    3. BACKWARD PASS:   Compute gradients for every weight -> Update all weights
```

### Step 1: Forward Pass
Data flows forward through the network. Each layer: `output = activation(W * input + b)`. By the final layer, you have a prediction — a class probability, a number, a label.

### Step 2: Calculate Loss
The **loss function** quantifies: "How wrong is my prediction?" A good loss function gives:
- Small values when predictions are close to correct
- Large values when predictions are far from correct
- Smooth gradients (differentiable everywhere)

### Step 3: Backward Pass (Backpropagation)
This is the mathematical miracle of deep learning. Using the **chain rule** from calculus, we compute:

```
d(loss)/d(every_weight_in_the_entire_network)
```

This gradient tells us: "If I increase this specific weight by a tiny amount, how much does the loss change?" Then we update:

```
weight_new = weight_old - learning_rate * gradient
```

**Why "back" propagation?** Gradients flow BACKWARD through the network: Output layer -> Last hidden layer -> ... -> First hidden layer. Each layer's gradients depend on the next layer's gradients, in a chain reaction from output to input.

## 2. Loss Functions — Defining What "Wrong" Means

The loss function DEFINES your model's goal. If you pick the wrong loss, your model optimizes for the wrong thing.

| Loss Function | Formula | Used For | Example |
|--------------|---------|----------|---------|
| **MSE** | mean((y - y_pred)^2) | Regression | Predict house price |
| **Binary Cross-Entropy** | -[y*log(p) + (1-y)*log(1-p)] | Binary classification | Spam detection |
| **Categorical Cross-Entropy** | -sum(y_i * log(p_i)) | Multi-class (one-hot) | MNIST digits |
| **Sparse Categorical CE** | Same as above | Multi-class (integer labels) | MNIST (simpler — just use 0,1,2...) |

### Binary Cross-Entropy — A Closer Look

Imagine the model outputs p=0.9 for a positive example (y=1):
- Loss = -(1 * log(0.9) + 0 * log(0.1)) = -log(0.9) = 0.105 (small — good prediction!)

Now if the model outputs p=0.1 for the same positive example:
- Loss = -(1 * log(0.1) + 0 * log(0.9)) = -log(0.1) = 2.30 (large — bad prediction!)

The loss HEAVILY penalizes confident wrong predictions. This is why models learn to be both accurate AND well-calibrated.

## 3. Backpropagation — The Chain Rule in Action

The chain rule from calculus: If a depends on b, and b depends on c, then:

```
da/dc = da/db * db/dc
```

In a neural network, the loss depends on the output layer, which depends on the hidden layer, which depends on the input:

```
dL/dw1 = dL/d(output) * d(output)/d(hidden) * d(hidden)/d(input) * d(input)/dw1
```

Each term is computed as the error propagates backward. The gradient for WEIGHT w1 depends on:
1. How much the loss changes with the output (error signal)
2. How much the output changes with the hidden layer
3. How much the hidden layer changes with the input
4. How much the input changes with w1

**This is why activation functions must be differentiable.** ReLU is differentiable everywhere except at 0 (where we define the gradient as 0). Sigmoid and tanh are differentiable everywhere.

## 4. Optimizers — Smarter Ways to Update Weights

Basic gradient descent: w = w - lr * gradient. But there are smarter ways.

| Optimizer | Key Innovation | When to Use |
|-----------|---------------|-------------|
| **SGD** | Basic: w -= lr * dL/dw | Rarely used alone |
| **SGD + Momentum** | Remembers past direction (like a heavy ball rolling downhill) | When SGD oscillates |
| **Adam** | Adaptive LR per parameter + momentum | **DEFAULT — 95% of the time** |
| **RMSprop** | Divides LR by recent gradient magnitude | RNNs, LSTMs |
| **AdamW** | Adam with decoupled weight decay | Modern PyTorch default |

### Why Adam is the Default

Adam adapts the learning rate INDIVIDUALLY for every parameter. Parameters that consistently get large gradients get their LR reduced (prevents overshooting). Parameters that get tiny gradients get their LR increased (helps them learn). This is like each neuron having its own customized learning speed.

### The Learning Rate — Most Critical Hyperparameter

| LR | What Happens | When You'd See This |
|:---:|---|------|
| 0.00001 | Training crawls — takes forever | You walk away and come back hours later |
| 0.0001 | Slow but stable convergence | Good for fine-tuning |
| **0.001** | **Sweet spot for Adam** | **Start here** |
| 0.01 | Fast, may overshoot or oscillate | Loss bounces up and down |
| 0.1 | Loss explodes to NaN | Model completely broken |

**Analogy:** Learning rate = step size while walking downhill in fog.
- Too small (tiptoe): You'll reach the bottom... eventually. Bring snacks.
- Too large (leap): You might jump over the valley or fall off a cliff.
- Just right: Confident strides downhill, reaching the bottom quickly and safely.

## 5. Overfitting — The #1 Problem in Deep Learning

### How to Detect It
- Training loss keeps decreasing (getting better)
- Validation loss starts INCREASING (getting worse!)
- The gap between them widens
- This is the model MEMORIZING training data instead of LEARNING patterns

### How to Fix It

**1. Early Stopping:** Stop training when validation loss stops improving. Restore the best weights. This alone prevents 80% of overfitting.

**2. Dropout:** Randomly DISABLE a fraction of neurons during training (typically 20-50%). Forces the network to build redundancy — no single neuron can be critical. Analogy: A sports team where any player might sit out any game forces the whole team to be strong.

**3. Reduce Model Size:** Fewer layers, fewer neurons. A smaller model has less capacity to memorize.

**4. Data Augmentation:** Create more training data by transforming existing data (flip images, add noise). Only works for certain data types.

**5. L1/L2 Regularization:** Add a penalty for large weights directly to the loss function. Large weights = complex model = more likely to overfit.

---

# PART 2: PRACTICAL

## 6. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {bool(tf.config.list_physical_devices("GPU"))}')


In [ ]:
# Generate a classification dataset with 3000 samples, 20 features
X, y = make_classification(
    n_samples=3000,
    n_features=20,
    n_informative=12,
    random_state=42
)

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Training data: {X_train_s.shape[0]} samples, {X_train_s.shape[1]} features')
print(f'Test data:     {X_test_s.shape[0]} samples')
print(f'Class balance (train): {y_train.mean():.1%} positive')
print(f'Data ready for neural network training!')


In [ ]:
# Model builder function — reuse for all experiments
def build_model():
    return keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(20,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

print('Model builder function defined.')
print('Architecture: 20 -> 128 -> 64 -> 32 -> 1 (sigmoid)')
print(f'This model has {build_model().count_params():,} trainable parameters.')


## 7. Compare All Optimizers

In [ ]:
optimizers = {
    'SGD (lr=0.01)': keras.optimizers.SGD(learning_rate=0.01),
    'SGD + Momentum': keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'Adam (lr=0.001)': keras.optimizers.Adam(learning_rate=0.001),
    'Adam (lr=0.01)': keras.optimizers.Adam(learning_rate=0.01),
    'RMSprop': keras.optimizers.RMSprop(learning_rate=0.001),
}

histories = {}
for name, opt in optimizers.items():
    model = build_model()
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

    history = model.fit(
        X_train_s, y_train,
        epochs=50,
        validation_split=0.2,
        verbose=0
    )
    histories[name] = history

    _, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
    final_val_loss = history.history['val_loss'][-1]
    print(f'{name:25s} | Test Acc: {test_acc:.3f} | Final Val Loss: {final_val_loss:.4f}')


In [ ]:
# Visual comparison
plt.figure(figsize=(12, 6))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
for (name, hist), color in zip(histories.items(), colors):
    plt.plot(hist.history['val_accuracy'], label=name,
            linewidth=2, color=color)
plt.xlabel('Epoch', fontsize=13)
plt.ylabel('Validation Accuracy', fontsize=13)
plt.title('Optimizer Comparison — Adam Clearly Wins', fontsize=15, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Learning Rate — Visualize the Difference

In [ ]:
learning_rates = [0.0001, 0.001, 0.01, 0.1]
lr_histories = {}

for lr in learning_rates:
    model = build_model()
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    history = model.fit(
        X_train_s, y_train,
        epochs=50,
        validation_split=0.2,
        verbose=0
    )
    lr_histories[lr] = history

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for lr, hist in lr_histories.items():
    axes[0].plot(hist.history['loss'], label=f'lr={lr}', linewidth=2)
    axes[1].plot(hist.history['val_loss'], label=f'lr={lr}', linewidth=2)

axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Effect of Learning Rate on Training', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary
print('LEARNING RATE SUMMARY:')
for lr, hist in lr_histories.items():
    final = hist.history['val_loss'][-1]
    status = 'TOO SLOW' if lr == 0.0001 else 'BEST' if lr == 0.001 else 'UNSTABLE' if lr == 0.01 else 'BROKEN'
    print(f'  lr={lr:.4f} -> Final Val Loss: {final:.4f} [{status}]')


## 9. Early Stopping — Automatic Overfitting Prevention

In [ ]:
model = build_model()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping: stop when validation loss hasn't improved for 8 epochs
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train_s, y_train,
    epochs=200,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

best_epoch = np.argmin(history.history['val_loss']) + 1
stopped_epoch = len(history.history['loss'])

print(f'\nTraining stopped at epoch {stopped_epoch}')
print(f'Best model was at epoch {best_epoch}')
print(f'Best validation loss: {min(history.history["val_loss"]):.4f}')

# Visualize
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.axvline(x=best_epoch-1, color='green', linestyle='--', linewidth=2,
            label=f'Best Model (Epoch {best_epoch})')

# Highlight overfitting zone
if stopped_epoch > best_epoch:
    plt.fill_between(range(best_epoch-1, stopped_epoch), 0, 1,
                    alpha=0.1, color='red')
    plt.text(best_epoch + 5, 0.8, 'OVERFITTING', fontsize=11,
            color='red', fontweight='bold')

plt.xlabel('Epoch', fontsize=13)
plt.ylabel('Loss', fontsize=13)
plt.title('Early Stopping Automatically Prevents Overfitting', fontsize=15, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Dropout + Batch Normalization — Regularization Battle

In [ ]:
# Train 3 models with different regularization on a SMALL dataset (500 samples)
# Small data = more overfitting = regularization effect more visible
n_small = 500

# Model 1: No regularization
model_plain = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(20,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model_plain.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
hist_plain = model_plain.fit(
    X_train_s[:n_small], y_train[:n_small],
    epochs=100, validation_split=0.2, verbose=0
)

# Model 2: With Dropout
model_dropout = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(20,)),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model_dropout.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
hist_dropout = model_dropout.fit(
    X_train_s[:n_small], y_train[:n_small],
    epochs=100, validation_split=0.2, verbose=0
)

# Model 3: Dropout + BatchNorm
model_best = keras.Sequential([
    layers.Dense(128, input_shape=(20,)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.5),
    layers.Dense(64),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.5),
    layers.Dense(32),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dense(1, activation='sigmoid')
])
model_best.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
hist_best = model_best.fit(
    X_train_s[:n_small], y_train[:n_small],
    epochs=100, validation_split=0.2, verbose=0
)

# Compare
_, acc_plain = model_plain.evaluate(X_test_s, y_test, verbose=0)
_, acc_dropout = model_dropout.evaluate(X_test_s, y_test, verbose=0)
_, acc_best = model_best.evaluate(X_test_s, y_test, verbose=0)

print(f'REGULARIZATION COMPARISON (trained on only {n_small} samples):')
print(f'  No Regularization:        Test Acc = {acc_plain:.3f}')
print(f'  + Dropout:                Test Acc = {acc_dropout:.3f}')
print(f'  + Dropout + BatchNorm:    Test Acc = {acc_best:.3f}')


In [ ]:
# Visual comparison
plt.figure(figsize=(12, 5))
plt.plot(hist_plain.history['val_loss'], label='No Regularization', linewidth=2)
plt.plot(hist_dropout.history['val_loss'], label='+ Dropout (0.5)', linewidth=2)
plt.plot(hist_best.history['val_loss'], label='+ Dropout + BatchNorm', linewidth=2)
plt.xlabel('Epoch', fontsize=13)
plt.ylabel('Validation Loss', fontsize=13)
plt.title('Regularization Strategies on Small Data (500 samples)', fontsize=15, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('With regularization, validation loss stays LOWER for LONGER.')
print('Without regularization, the model quickly overfits and loss goes UP.')


---

# PART 3: EXERCISES

In [ ]:
# EXERCISE 1: Find the best learning rate for Adam
print('SEARCHING FOR BEST LEARNING RATE...')
best_lr, best_acc = 0, 0
for lr in [0.0001, 0.0005, 0.001, 0.005, 0.01]:
    model = build_model()
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    model.fit(X_train_s, y_train, epochs=30, validation_split=0.2, verbose=0)
    _, acc = model.evaluate(X_test_s, y_test, verbose=0)
    print(f'  lr={lr:.4f} -> Test Acc: {acc:.4f}')
    if acc > best_acc:
        best_lr, best_acc = lr, acc

print(f'\nBEST: lr={best_lr} with accuracy {best_acc:.4f}')


In [ ]:
# EXERCISE 2: ReduceLROnPlateau — Adaptive learning rate
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,          # Halve the LR
    patience=5,          # Wait 5 epochs before reducing
    min_lr=1e-6,         # Don't go below this
    verbose=1
)

model = build_model()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train_s, y_train,
    epochs=80,
    validation_split=0.2,
    callbacks=[reduce_lr],
    verbose=0
)

# Show when LR was reduced
print(f'\nTraining complete! Final LR reductions visible above.')
print(f'Test accuracy: {model.evaluate(X_test_s, y_test, verbose=0)[1]:.4f}')


In [ ]:
# EXERCISE 3: Train a deeper network with all techniques combined
model_final = keras.Sequential([
    layers.Dense(256, input_shape=(20,)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),

    layers.Dense(128),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),

    layers.Dense(64),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model_final.compile(
    optimizer=keras.optimizers.Adam(learning_rate=best_lr),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Combine early stopping + reduce LR
callbacks_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

history_final = model_final.fit(
    X_train_s, y_train,
    epochs=200,
    validation_split=0.2,
    callbacks=callbacks_list,
    verbose=1
)

test_loss, test_acc = model_final.evaluate(X_test_s, y_test, verbose=0)
print(f'\nFINAL MODEL:')
print(f'  Test Accuracy: {test_acc:.4f}')
print(f'  Test Loss:     {test_loss:.4f}')
print(f'  Architecture:  256 -> 128 -> 64 -> 32 -> 1')
print(f'  Regularization: BatchNorm + Dropout + Early Stopping + Adaptive LR')


## Key Takeaways — Day 63

- The training loop: **Forward Pass -> Loss -> Backprop -> Update** (repeat)
- **Binary Cross-Entropy** for binary classification, **MSE** for regression
- **Backpropagation** = chain rule applied to EVERY weight in the network
- **Adam** is the default optimizer — adaptive, fast, works out of the box
- **Learning rate 0.001** is the best starting point for Adam
- **Early Stopping** automatically prevents overfitting (restore best weights)
- **Dropout** randomly disables neurons during training (creates redundancy)
- **BatchNorm** stabilizes training and allows higher learning rates
- **ReduceLROnPlateau** automatically lowers LR when progress stalls

**Tomorrow:** MNIST — Build your first complete neural network project!